# Dataset Overview

This notebook documents the dataset used to model how footballer performance qualities change after within-league transfers. It covers data sources, filtering pipeline, feature definitions, and the final modelling-ready dataset.

## 1. Data Sources

| Source | What it provides |
|--------|------------------|
| **Twelve Football (Wyscout)** | Player performance metrics, 20 Twelve Quality Scores, team tactical stats |
| **Transfermarkt** | Transfer records, fees, market values |

The raw data was processed into two parquet files:

- `within_league_transfers_v5.parquet` — One row per player-transfer, with pre/post qualities and team stats
- `team_qualities.parquet` — One row per team-competition-season, with 7 tactical style dimensions

In [1]:
import pandas as pd
import numpy as np

DATA_DIR = "../../../../thesis_data/processed_data/thesis_model_dataset/active/"

df = pd.read_parquet(DATA_DIR + "within_league_transfers_v5.parquet")
print(f"Full dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")

Full dataset: 18,065 rows, 120 columns


## 2. Filtering Pipeline

The master dataset (262K rows) was filtered through the earlier notebooks (01-05) to produce v5:

| Step | Filter | Effect |
|------|--------|--------|
| 1 | Same league (`from_competition == to_competition`) | Within-league transfers only |
| 2 | Different team (`from_team_id != to_team_id`) | Exclude same-team re-entries |
| 3 | Same position (`from_position == to_position`) | Consistent role comparison |
| 4 | 900+ minutes both sides | Sufficient sample in both spells |
| 5 | No goalkeepers | 5 outfield positions only |

Result: **~18,000 within-league transfers** across 5 positions.

In [2]:
print("Position breakdown:")
print(df["from_position"].value_counts().to_string())
print(f"\nSeasons: {sorted(df['from_season'].unique())}")

Position breakdown:
from_position
Midfielder          5230
Central Defender    4766
Full Back           3812
Striker             2422
Winger              1835

Seasons: [2018, 2019, 2020, 2021, 2022, 2023, 2024]


## 3. Midfielder Subset

All models in this thesis focus on **Midfielders** as the target position.

In [3]:
mf = df[df["from_position"] == "Midfielder"].copy()
print(f"Midfielders: {mf.shape[0]:,} rows")

Midfielders: 5,230 rows


## 4. Feature Definitions

### 4.1 Player Quality Scores (20 total, 17 used for Midfielders)

Twelve Football computes 20 composite quality scores per player from position-adjusted z-scores. Each transfer row has `from_Q` (pre-transfer) and `to_Q` (post-transfer) for all 20.

For midfielders, **3 qualities are excluded** due to >50% nulls:
- Chance prevention (100% null)
- Territorial dominance (100% null)
- Poaching (75% null)

The remaining **17 qualities** are our modelling targets.

In [4]:
QUALITIES = [
    "Active defence", "Aerial threat", "Box threat", "Composure",
    "Defensive heading", "Dribbling", "Effectiveness", "Finishing",
    "Hold-up play", "Intelligent defence", "Involvement",
    "Passing quality", "Pressing", "Progression",
    "Providing teammates", "Run quality", "Winning duels",
]

from_pq = [f"from_{q}" for q in QUALITIES]
to_pq = [f"to_{q}" for q in QUALITIES]

# Show null rates for the excluded qualities
excluded = ["Chance prevention", "Territorial dominance", "Poaching"]
for q in excluded:
    null_pct = mf[f"from_{q}"].isna().mean() * 100
    print(f"{q}: {null_pct:.1f}% null")

print(f"\n17 modelling qualities null rates:")
null_rates = mf[from_pq].isna().mean().sort_values(ascending=False)
print(null_rates[null_rates > 0].to_string())
print(f"\nQualities with 0% nulls: {(null_rates == 0).sum()}")

Chance prevention: 100.0% null
Territorial dominance: 100.0% null
Poaching: 75.4% null

17 modelling qualities null rates:
from_Effectiveness        0.002868
from_Finishing            0.002868
from_Aerial threat        0.001530
from_Defensive heading    0.000382
from_Dribbling            0.000191

Qualities with 0% nulls: 12


### 4.2 Team Tactical Qualities (7 dimensions)

Each team-season is scored on 7 tactical style dimensions, computed from 21 underlying z-scored stats:

| Dimension | What it captures |
|-----------|------------------|
| Defence | Defensive solidity and organisation |
| Defensive Transition | Speed and effectiveness of recovering possession |
| Attacking Transition | Counter-attacking and fast-break play |
| Attack | Sustained possession-based attacking |
| Penetration | Ability to break into the box |
| Chance Creation | Creating shooting opportunities |
| Outcome | Converting chances into goals/points |

In the dataset:
- `from_q_proj_*` — Origin team qualities, **projected** to the destination season's league distribution
- `to_q_*` — Destination team qualities, expressed in their own season's league distribution

Both are z-scored against the same reference (destination league-season), making them directly comparable.

In [5]:
TEAM_Q = ["DEFENCE", "DEFENSIVE_TRANSITION", "ATTACKING_TRANSITION",
          "ATTACK", "PENETRATION", "CHANCE_CREATION", "OUTCOME"]

from_tq = [f"from_q_proj_{q}" for q in TEAM_Q]
to_tq = [f"to_q_{q}" for q in TEAM_Q]

print("Team quality null rates (midfielders):")
tq_nulls = mf[from_tq + to_tq].isna().mean()
print(tq_nulls[tq_nulls > 0].to_string() if (tq_nulls > 0).any() else "All complete")

Team quality null rates (midfielders):
from_q_proj_DEFENCE                 0.049713
from_q_proj_DEFENSIVE_TRANSITION    0.049713
from_q_proj_ATTACKING_TRANSITION    0.049713
from_q_proj_ATTACK                  0.049713
from_q_proj_PENETRATION             0.049713
from_q_proj_CHANCE_CREATION         0.049713
from_q_proj_OUTCOME                 0.049713
to_q_DEFENCE                        0.033078
to_q_DEFENSIVE_TRANSITION           0.033078
to_q_ATTACKING_TRANSITION           0.033078
to_q_ATTACK                         0.033078
to_q_PENETRATION                    0.033078
to_q_CHANCE_CREATION                0.033078
to_q_OUTCOME                        0.033078


### 4.3 Control Variables

- `player_season_age` — Player age at transfer
- `from_Minutes` / `to_Minutes` — Minutes played before and after transfer

### 4.4 Derived Features (Deltas)

Used in Models 4a-4c:

- **Delta player quality:** `delta_Q_i = to_Q_i - from_Q_i` (what we predict)
- **Delta team quality:** `delta_team_Q_k = to_q_Q_k - from_q_proj_Q_k` (change in tactical environment)
- **Delta minutes:** `delta_Minutes = to_Minutes - from_Minutes`

## 5. Clean Dataset & Train/Test Split

After dropping rows with nulls in any of the modelling columns, we split **temporally**:
- **Train:** seasons 2018-2023
- **Test:** season 2024

In [6]:
# All columns needed across the 6 models
delta_tq = [f"delta_team_{q}" for q in TEAM_Q]
delta_pq = [f"delta_{q}" for q in QUALITIES]

# Construct deltas
for q in QUALITIES:
    mf[f"delta_{q}"] = mf[f"to_{q}"] - mf[f"from_{q}"]
for q in TEAM_Q:
    mf[f"delta_team_{q}"] = mf[f"to_q_{q}"] - mf[f"from_q_proj_{q}"]
mf["delta_Minutes"] = mf["to_Minutes"] - mf["from_Minutes"]

# Columns needed for all 6 models
all_cols = (from_pq + to_pq + from_tq + to_tq + delta_tq + delta_pq
            + ["player_season_age", "delta_Minutes", "from_season"])

mf_clean = mf[all_cols].dropna()

train = mf_clean[mf_clean["from_season"] <= 2023]
test = mf_clean[mf_clean["from_season"] == 2024]

print(f"Clean midfielders: {len(mf_clean):,}")
print(f"Train (2018-2023): {len(train):,}")
print(f"Test (2024):       {len(test):,}")
print(f"\nSeasons in train: {sorted(train['from_season'].unique())}")

Clean midfielders: 4,848
Train (2018-2023): 4,383
Test (2024):       465

Seasons in train: [2018, 2019, 2020, 2021, 2022, 2023]


## 6. Summary: What Goes Into Each Model

| Model | Features | Target | # Features |
|-------|----------|--------|------------|
| **1. Naive** | `from_Q_i` | `to_Q_i` | 1 per quality |
| **2. All Pre-Qualities** | `from_Q_1 ... from_Q_17` | `to_Q_i` | 17 |
| **3. Pre-Q + Teams** | `from_Q_1..17` + `from_team_1..7` + `to_team_1..7` | `to_Q_i` | 31 |
| **4a. Delta Team** | `delta_team_1..7` | `delta_Q_i` | 7 |
| **4b. Pre-Q + Delta Team** | `from_Q_1..17` + `delta_team_1..7` | `delta_Q_i` | 24 |
| **4c. + Controls** | `from_Q_1..17` + `delta_team_1..7` + `age` + `delta_min` | `delta_Q_i` | 26 |

Each model is fitted **17 times** (once per quality), all using OLS with statsmodels.